# Speed Benchmarks: BitNet vs Ollama

This notebook runs speed benchmarks comparing native 1.58-bit (ternary) BitNet models against post-training quantized models served via Ollama.

**Metrics:** prefill tok/s, decode tok/s, peak RAM (MB), wall time (s)  
**Protocol:** 2 warm-up runs (discarded) + 3 recorded trials per configuration

In [ ]:
import sys, os

# Add project root to path
PROJECT_ROOT = os.path.dirname(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from configs.benchmark_config import (
    BITNET_MODELS, OLLAMA_MODELS, PROMPT_CONFIGS,
    THREAD_COUNTS, WARMUP_RUNS, RECORDED_TRIALS,
    CSV_COLUMNS, SPEED_CSV, BITNET_BENCHMARK_SCRIPT,
    RESULTS_DIR,
)
from utils.ollama_runner import run_ollama_benchmark, preload_model
from utils.bitnet_runner import run_bitnet_benchmark

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {SPEED_CSV}")

## 1. Verify Ollama Models

Check which models are available and their quantization levels.

In [ ]:
import subprocess

# List available models
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print("Available Ollama models:")
print(result.stdout)

# Check quant details for each model we need
for model_name in OLLAMA_MODELS:
    print(f"\n--- {model_name} ---")
    result = subprocess.run(
        ["ollama", "show", model_name, "--modelfile"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        # Print just the first few lines with quant info
        for line in result.stdout.split("\n")[:10]:
            print(line)
    else:
        print(f"  NOT INSTALLED — run: ollama pull {model_name}")

## 2. Pull Missing Ollama Models

Run this cell if any models from the previous check are missing.

In [ ]:
for model_name in OLLAMA_MODELS:
    print(f"Pulling {model_name}...")
    result = subprocess.run(
        ["ollama", "pull", model_name],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f"  ✓ {model_name} ready")
    else:
        print(f"  ✗ Failed: {result.stderr}")

## 3. Run Ollama Benchmarks

In [ ]:
rows = []

for model_name, model_info in OLLAMA_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    # Preload model into memory
    print("Preloading model...")
    preload_model(model_name)

    for config_name, config in PROMPT_CONFIGS.items():
        prompt_tokens = config["prompt_tokens"]
        gen_tokens = config["gen_tokens"]

        print(f"\n  Config: {config_name} (p={prompt_tokens}, g={gen_tokens})")

        # Warm-up runs
        for w in range(WARMUP_RUNS):
            print(f"    Warm-up {w+1}/{WARMUP_RUNS}...", end=" ")
            try:
                run_ollama_benchmark(model_name, prompt_tokens, gen_tokens)
                print("done")
            except Exception as e:
                print(f"failed: {e}")

        # Recorded trials
        for trial in range(1, RECORDED_TRIALS + 1):
            print(f"    Trial {trial}/{RECORDED_TRIALS}...", end=" ")
            try:
                result = run_ollama_benchmark(model_name, prompt_tokens, gen_tokens)
                row = {
                    "comparison": model_info["comparison"],
                    "model_name": model_name,
                    "param_count": model_info["param_count"],
                    "quant_method": model_info["quant_method"],
                    "runner": "ollama",
                    "prompt_tokens": prompt_tokens,
                    "gen_tokens": gen_tokens,
                    "threads": "default",
                    "trial": trial,
                    "prefill_toks": result["prefill_toks"],
                    "decode_toks": result["decode_toks"],
                    "peak_ram_mb": result["peak_ram_mb"],
                    "idle_ram_mb": result["idle_ram_mb"],
                    "wall_time_s": result["wall_time_s"],
                }
                rows.append(row)
                print(f"prefill={result['prefill_toks']:.1f} tok/s, "
                      f"decode={result['decode_toks']:.1f} tok/s, "
                      f"RAM={result['peak_ram_mb']:.0f} MB")
            except Exception as e:
                print(f"failed: {e}")

# Save results
df_ollama = pd.DataFrame(rows, columns=CSV_COLUMNS)
df_ollama.to_csv(SPEED_CSV, index=False)
print(f"\nSaved {len(rows)} rows to {SPEED_CSV}")
df_ollama.head()

## 4. Run BitNet Benchmarks

**Prerequisites:** BitNet repo cloned and model(s) downloaded + prepared with `setup_env.py`.  
Skip this section if BitNet is not set up yet.

In [ ]:
# Check BitNet availability
from configs.benchmark_config import BITNET_REPO

bitnet_available = os.path.exists(BITNET_BENCHMARK_SCRIPT)
print(f"BitNet repo path: {BITNET_REPO}")
print(f"Benchmark script exists: {bitnet_available}")

if bitnet_available:
    for name, info in BITNET_MODELS.items():
        exists = os.path.exists(info["path"])
        print(f"  {name}: {'✓' if exists else '✗ NOT FOUND'} — {info['path']}")
else:
    print("\nBitNet not found. To set up:")
    print(f"  cd {os.path.dirname(BITNET_REPO)}")
    print("  git clone --recursive https://github.com/microsoft/BitNet.git")
    print("  cd BitNet")
    print("  pip install -r requirements.txt")
    print("  huggingface-cli download microsoft/BitNet-b1.58-2B-4T-gguf --local-dir models/BitNet-b1.58-2B-4T")
    print("  python setup_env.py -md models/BitNet-b1.58-2B-4T -q i2_s")

In [ ]:
if not bitnet_available:
    print("Skipping BitNet benchmarks — repo not found.")
else:
    bitnet_rows = []

    for model_name, model_info in BITNET_MODELS.items():
        if not os.path.exists(model_info["path"]):
            print(f"Skipping {model_name} — model file not found")
            continue

        print(f"\n{'='*60}")
        print(f"Model: {model_name}")
        print(f"{'='*60}")

        for config_name, config in PROMPT_CONFIGS.items():
            prompt_tokens = config["prompt_tokens"]
            gen_tokens = config["gen_tokens"]

            for threads in THREAD_COUNTS:
                print(f"\n  Config: {config_name}, threads={threads}")

                # Warm-up
                for w in range(WARMUP_RUNS):
                    print(f"    Warm-up {w+1}/{WARMUP_RUNS}...", end=" ")
                    try:
                        run_bitnet_benchmark(
                            model_info["path"], prompt_tokens, gen_tokens,
                            threads, BITNET_BENCHMARK_SCRIPT,
                        )
                        print("done")
                    except Exception as e:
                        print(f"failed: {e}")

                # Recorded trials
                for trial in range(1, RECORDED_TRIALS + 1):
                    print(f"    Trial {trial}/{RECORDED_TRIALS}...", end=" ")
                    try:
                        result = run_bitnet_benchmark(
                            model_info["path"], prompt_tokens, gen_tokens,
                            threads, BITNET_BENCHMARK_SCRIPT,
                        )
                        row = {
                            "comparison": model_info["comparison"],
                            "model_name": model_name,
                            "param_count": model_info["param_count"],
                            "quant_method": model_info["quant_method"],
                            "runner": "bitnet.cpp",
                            "prompt_tokens": prompt_tokens,
                            "gen_tokens": gen_tokens,
                            "threads": threads,
                            "trial": trial,
                            "prefill_toks": result["prefill_toks"],
                            "decode_toks": result["decode_toks"],
                            "peak_ram_mb": result["peak_ram_mb"],
                            "idle_ram_mb": 0,
                            "wall_time_s": result["wall_time_s"],
                        }
                        bitnet_rows.append(row)
                        print(f"prefill={result['prefill_toks']:.1f} tok/s, "
                              f"decode={result['decode_toks']:.1f} tok/s, "
                              f"RAM={result['peak_ram_mb']:.0f} MB")
                    except Exception as e:
                        print(f"failed: {e}")

    # Append to CSV
    if bitnet_rows:
        df_bitnet = pd.DataFrame(bitnet_rows, columns=CSV_COLUMNS)
        if os.path.exists(SPEED_CSV):
            df_bitnet.to_csv(SPEED_CSV, mode="a", header=False, index=False)
        else:
            df_bitnet.to_csv(SPEED_CSV, index=False)
        print(f"\nAppended {len(bitnet_rows)} BitNet rows to {SPEED_CSV}")

## 5. Quick Results Summary

In [ ]:
if os.path.exists(SPEED_CSV):
    df = pd.read_csv(SPEED_CSV)
    print(f"Total rows: {len(df)}")
    print(f"Models tested: {df['model_name'].unique().tolist()}")
    print()

    summary = df.groupby("model_name").agg(
        avg_prefill=("prefill_toks", "mean"),
        avg_decode=("decode_toks", "mean"),
        avg_ram=("peak_ram_mb", "mean"),
    ).round(1)
    print(summary)
else:
    print("No results yet — run benchmarks first.")